# Introduction to TensorFlow & Keras — Part 2
### The Functional API, branching models, and multi-input problems

Picking up right where Part 1 left off. There, you built and trained
`digit_classifier` — a single-input, single-output stack — using the
Sequential API, and were asked to imagine a case where Sequential
wouldn't be enough. This notebook builds that case for real.

### Code rules for this notebook
1. Keep all your imports in the next cell.
2. Give variables names that say what they hold.
3. Add type hints to every function you write.
4. Every model must go through `train_and_evaluate()` once you've written it below.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Any, Dict

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.mnist.load_data()
X_train = X_train_raw.reshape(-1, 28 * 28).astype("float32") / 255.0
X_test = X_test_raw.reshape(-1, 28 * 28).astype("float32") / 255.0

print(f"X_train shape: {X_train.shape}")

X_train shape: (60000, 784)


## Step 1: The Functional API, syntax first

Recap: in the Functional API you don't hand Keras a list. You create
an `Input`, then call each layer directly on a tensor, passing the
result forward yourself. The model is then defined by which tensors
are its inputs and outputs.

📖 [The Functional API (Keras docs)](https://www.tensorflow.org/guide/keras/functional_api)

In [2]:
inputs = keras.Input(shape=(784,))
x = layers.Dense(64, activation="relu")(inputs)  # the Input tensor feeds the first layer
x = layers.Dropout(0.2)(x)  # each layer consumes the previous tensor in the chain
outputs = layers.Dense(10, activation="softmax")(x)  # the hidden tensor feeds the output layer

functional_classifier = keras.Model(inputs=inputs, outputs=outputs)  # these two tensors define the graph
functional_classifier.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,890 (198.79 KB)

 Trainable params: 50,890 (198.79 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
try:
    keras.utils.plot_model(functional_classifier, show_shapes=True, show_layer_names=True)
except ImportError as graphviz_error:
    # plot_model draws the graph with the external graphviz `dot` binary. Colab
    # ships it; where it is missing, fall back to the text summary rather than
    # letting one missing system package break the whole notebook.
    print(f"plot_model needs graphviz, which is missing here ({graphviz_error}).")
    print("Showing the text summary instead -- run this in Colab to get the diagram.")
    functional_classifier.summary()

plot_model needs graphviz, which is missing here (You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.).
Showing the text summary instead -- run this in Colab to get the diagram.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,890 (198.79 KB)

 Trainable params: 50,890 (198.79 KB)

 Non-trainable params: 0 (0.00 B)

**🧠 Your turn:** Compare this `summary()` to `digit_classifier`'s from
Part 1. Same layers, same shapes — what's actually different between
the two versions?

_Write your answer here:_ The two summaries are essentially identical -- the same three layers, the same parameter counts, the same output shapes. Nothing about the *network* changed. What changed is how the model was **declared**. `Sequential` takes an ordered list of layers and wires each one to the previous implicitly; it is a model that happens to be a chain. The Functional version builds an explicit graph: `inputs` is a real tensor object, each layer is *called* on a tensor, and the model is defined by naming the input and output tensors. That is strictly more expressive for the same amount of code, which is why the rest of this notebook can branch and merge without leaving the API.

## Step 2: One function you'll reuse for the rest of this notebook

Same principle as before: write it once, reuse it for every model
below.

In [4]:
def train_and_evaluate(
    model: keras.Model,
    x: Any,
    y: Any,
    epochs: int = 30,
    batch_size: int = 128,
) -> Dict[str, Any]:
    # Compiles-agnostic: assumes `model` is already compiled.
    # Trains with a 10% validation split and early stopping, then
    # reports how it did. Reused for every model in this notebook.
    #
    # x, y: training inputs and labels. x can be a single array, or a
    # list of arrays for models with more than one input.
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True
    )

    history = model.fit(
        x, y,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=0,
    )

    return {
        "history": history.history,
        "final_train_accuracy": history.history["accuracy"][-1],
        "final_val_accuracy": history.history["val_accuracy"][-1],
        "epochs_run": len(history.history["loss"]),
    }

In [5]:
functional_classifier.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
functional_results = train_and_evaluate(functional_classifier, X_train, y_train)
print(f"Final validation accuracy: {functional_results['final_val_accuracy']:.4f}")

Final validation accuracy: 0.9785


## Step 3: A real multi-input problem — image + text

Here's a genuine reason to want two separate inputs. Picture a bank's
check-processing pipeline: each check has a handwritten digit (the
image) for the amount, *and* a separate typed field where the amount
was also written out in words — a real cross-checking feature actual
check-scanning systems use. The catch: that typed field gets OCR'd
automatically, and OCR sometimes misreads it.

Below, we simulate exactly that: a short text "hint" that agrees with
the true digit most of the time, but is deliberately wrong some of
the time — just like a noisy OCR read would be. (The simulation code
is given below — it's not the point of this step. What matters is
what comes next: building a model that actually takes both inputs.)

`Sequential` can't take two independent inputs like this. The
Functional API can: one `Input` for the image, one `Input` for the
text token, each with its own layers, merged with `Concatenate`
before the final prediction.

📖 [Embedding layer docs](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding) · [Multi-input/output models (Functional API guide)](https://www.tensorflow.org/guide/keras/functional_api#models_with_multiple_inputs_and_outputs) · [Concatenate layer docs](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Concatenate)

In [ ]:
DIGIT_WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
OCR_ERROR_RATE = 0.2  # how often the typed hint disagrees with the true digit

def make_noisy_text_hints(true_labels: np.ndarray, error_rate: float, seed: int) -> np.ndarray:
    # Simulates an OCR'd text field: returns one word-index per example,
    # matching the true label most of the time, but wrong `error_rate`
    # fraction of the time (a random *different* digit). Not the point
    # of this notebook -- just setup so Step 3 has two real inputs to work with.
    rng = np.random.default_rng(seed)
    is_wrong = rng.random(len(true_labels)) < error_rate
    random_digit = rng.integers(0, 10, size=len(true_labels))
    return np.where(is_wrong, random_digit, true_labels)

# Different seeds, so the test hints are independent noise rather than a
# replay of the training hints' random stream.
train_text_hints = make_noisy_text_hints(y_train, OCR_ERROR_RATE, seed=RANDOM_SEED)
test_text_hints = make_noisy_text_hints(y_test, OCR_ERROR_RATE, seed=RANDOM_SEED + 1)

print(f"Example: true digit = {y_train[0]}, typed hint = '{DIGIT_WORDS[train_text_hints[0]]}'")
print(f"Hint agrees with true label {np.mean(train_text_hints == y_train):.1%} of the time")

In [7]:
image_input = keras.Input(shape=(784,), name="image")
text_input = keras.Input(shape=(1,), name="text_hint")

image_branch = layers.Dense(64, activation="relu")(image_input)

text_branch = layers.Embedding(input_dim=len(DIGIT_WORDS), output_dim=8)(text_input)  # 10 possible words
text_branch = layers.Flatten()(text_branch)

combined = layers.Concatenate()([image_branch, text_branch])  # merge the image and text branches
outputs = layers.Dense(10, activation="softmax")(combined)

fusion_model = keras.Model(inputs=[image_input, text_input], outputs=outputs)
fusion_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_hint           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image (InputLayer)  │ (None, 784)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 8)      │         80 │ text_hint[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │     50,240 │ image[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8)         │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 72)        │          0 │ dense_2[0][0],    │
│ (Concatenate)       │                   │            │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 10)        │        730 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 51,050 (199.41 KB)

 Trainable params: 51,050 (199.41 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# A text summary lists layers top to bottom, but doesn't show branching.
# For a multi-input model, the picture makes the two-branches-merging
# shape obvious in a way the text summary above doesn't.
try:
    keras.utils.plot_model(fusion_model, show_shapes=True, show_layer_names=True)
except ImportError as graphviz_error:
    # plot_model draws the graph with the external graphviz `dot` binary. Colab
    # ships it; where it is missing, fall back to the text summary rather than
    # letting one missing system package break the whole notebook.
    print(f"plot_model needs graphviz, which is missing here ({graphviz_error}).")
    print("Showing the text summary instead -- run this in Colab to get the diagram.")
    fusion_model.summary()

plot_model needs graphviz, which is missing here (You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.).
Showing the text summary instead -- run this in Colab to get the diagram.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_hint           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image (InputLayer)  │ (None, 784)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 8)      │         80 │ text_hint[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │     50,240 │ image[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8)         │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 72)        │          0 │ dense_2[0][0],    │
│ (Concatenate)       │                   │            │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 10)        │        730 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 51,050 (199.41 KB)

 Trainable params: 51,050 (199.41 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
fusion_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
fusion_results = train_and_evaluate(fusion_model, x=[X_train, train_text_hints], y=y_train)
print(f"Combined (image + text) validation accuracy: {fusion_results['final_val_accuracy']:.4f}")

Combined (image + text) validation accuracy: 0.9830


**🧠 Your turn:** The text hint alone agrees with the true label about
80% of the time (since `OCR_ERROR_RATE = 0.2`), so a model that only
looked at the text could never beat ~80% accuracy. Did combining it
with the image push accuracy above that ceiling? What does that tell
you about combining two imperfect signals versus trusting either one
alone?

_Write your answer here:_ Yes — and by a clear margin. A model that only ever sees the text hint is capped near that ~81.9% ceiling, which is just the fraction of examples where the OCR'd hint happens to agree with the true digit. The image-only Functional classifier from Step 1 reached 0.9785 validation accuracy, and the combined image + text model reached **0.9830** — better than image-only, and far above the text-only ceiling.

What that tells me is that two imperfect signals can be worth more than either one alone, provided they carry **partly independent** information. The image is the strong signal; the text hint is weak and wrong about one time in five, but it is wrong in a different way, so it still contributes a little independent evidence at the margin. Giving each input its own branch and merging them with `Concatenate` lets the model learn how much to trust each one, instead of being forced to treat them identically.

Worth noticing: the gain over image-only is small (about +0.45 percentage points). That is what you would expect when one of the two signals is already at 98% — a noisy second opinion can only sharpen the decision boundary, not redefine it.

## Step 4: Branching and merging, more generally

Multiple inputs is one use of the Functional API's flexibility, but
the bigger idea is that a model is really just a **graph of layers**
— data can split into parallel branches and merge back together, even
with a single input. Below, the same image feeds two separate `Dense`
branches of different sizes, and their outputs are concatenated
before the final layer.

📖 [Manipulating complex graph topologies (Functional API guide)](https://www.tensorflow.org/guide/keras/functional_api#manipulate_complex_graph_topologies)

In [ ]:
branching_input = keras.Input(shape=(784,))

wide_branch = layers.Dense(128, activation="relu")(branching_input)
narrow_branch = layers.Dense(16, activation="relu")(branching_input)  # the SAME input feeds both branches

merged = layers.Concatenate()([wide_branch, narrow_branch])  # merge the two parallel branches
outputs = layers.Dense(10, activation="softmax")(merged)

branching_model = keras.Model(inputs=branching_input, outputs=outputs)
branching_model.summary()

In [11]:
try:
    keras.utils.plot_model(branching_model, show_shapes=True, show_layer_names=True)
except ImportError as graphviz_error:
    # plot_model draws the graph with the external graphviz `dot` binary. Colab
    # ships it; where it is missing, fall back to the text summary rather than
    # letting one missing system package break the whole notebook.
    print(f"plot_model needs graphviz, which is missing here ({graphviz_error}).")
    print("Showing the text summary instead -- run this in Colab to get the diagram.")
    branching_model.summary()

plot_model needs graphviz, which is missing here (You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.).
Showing the text summary instead -- run this in Colab to get the diagram.


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 784)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │    100,480 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 16)        │     12,560 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 144)       │          0 │ dense_4[0][0],    │
│ (Concatenate)       │                   │            │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 10)        │      1,450 │ concatenate_1[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 114,490 (447.23 KB)

 Trainable params: 114,490 (447.23 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
branching_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
branching_results = train_and_evaluate(branching_model, X_train, y_train)
print(f"Final validation accuracy: {branching_results['final_val_accuracy']:.4f}")

Final validation accuracy: 0.9797


**🧠 Your turn:** Could you build this exact branching model with
Sequential? Why or why not?

_Write your answer here:_ No. `Sequential` applies each layer in order to the output of the one before it, so the data can only ever move forward along a single path. This model does something a chain cannot express: `image_input` is consumed by **two** layers at once (`wide_branch` and `narrow_branch`), so the data splits into two parallel paths, and those two paths are then merged back together by `Concatenate`. `Sequential` has no way to name a layer's input, so it cannot reuse an earlier tensor or join two branches. That is precisely the limitation the Functional API exists to remove.

## Wrap-up

**Final question:** across this notebook you've now built the same
basic classifier three different ways — the plain Functional
equivalent of Part 1's model, a two-input version, and a branching
version. What's the one thing that stayed identical in every version,
and what changed?

_Write your answer here:_ What stayed identical is everything that defines the *learning problem*: the input shape `(784,)` flattened MNIST pixels, the same `sparse_categorical_crossentropy` loss, the same metrics, and the same classifier head -- a `Dense(10, activation="softmax")` producing one probability per digit. What changed is the **topology**, i.e. the shape of the graph between the input and the head: three versions used a straight line, two independent input streams merged at the end, and a single input fanning out into two parallel branches and merging back. The API you use to describe the model changed because the graph changed; the actual task never did. That is why all three land in a similar accuracy range on the same data.

## Going further

| Idea | What it roughly controls |
|---|---|
| `Conv2D` + `MaxPooling2D` | Convolutional layers — look at local patterns instead of a flat vector; usually the biggest jump on image data |
| Model subclassing | A third way to build models, writing the forward pass as regular Python code — more flexible than Functional, more work |
| Shared layers | Reusing the *same* layer object in two places in the graph, so it learns one shared representation |
| `model.save()` / `load_model()` | Saving a trained model to disk and reloading it later |

📖 [Convolutional layers (Keras image classification tutorial)](https://www.tensorflow.org/tutorials/images/cnn)